# AmSC Facility Access via RIG

Your AmSC PAT (from [`login-amsc.ipynb`](login-amsc.ipynb)) is enough to reach **RIG**, but each facility decides which credential it accepts.
For a facility call through RIG to succeed, two things must be true **for the project in your PAT**:

1. **RIG has a facility credential for you**: how depends on the facility's RIG tier (table below).
2. **Your project has resources at the facility**: without an allocation, the facility's IRI API denies access even with a
   valid credential.

Everything is per project: the Credential Vault stores credentials for the project that is active when you save them.
Use the same project that was selected in MyAmSC when you generated the PAT.

| Facility | RIG tier | Credential | Setup | Section |
|---|---|---|---|---|
| ESnet (`esnet-east`, `esnet-west`), PNNL (`pnnl`) | 1 | Your AmSC token (PAT) | Project active and resources at the site | 2 |
| NERSC (`nersc`) | 3 | Globus token authorized by NERSC | Save the token in the Credential Vault | 3 |
| ALCF (`alcf`) | 2 | ALCF login (via Globus), stored and refreshed by RIG | Connect in the Credential Vault | 4 |
| ORNL / OLCF (`olcf-open`, `olcf-moderate`) | 3 | S3M token from the ORNL website | Save the token in the Credential Vault | 5 |

Section 6 checks your access to every facility with your PAT. Run it again after each change.

## 1) Open the Credential Vault (RIG-UI)

Open **https://rig-ui.staging.american-science-cloud.org/credential-vault** and log in with your AmSC identity.

- The **Project** shown at the top must be the project of your PAT. Credentials are saved for that project only.
- Each facility card shows its tier and status:
  - **Direct via RIG**: Tier 1, nothing to configure here.
  - **Needs setup** / **Ready**: Tier 3, a facility token is missing / saved.
  - **Not connected** / **Connected**: Tier 2, the facility login is missing / linked.
- **Test with RIG** calls the facility through RIG with your current login.

![RIG-UI Credential Vault overview (project + facility cards)](images/rig-ui/credential-vault-overview.png)

## 2) ESnet and PNNL (Tier 1: AmSC token)

ESnet (`esnet-east`, `esnet-west`) and PNNL (`pnnl`) accept your **AmSC token directly**. There is nothing to configure in
the Credential Vault.

Requirements:
- **The project must be active**: the PAT must carry the project (`amsc_project_context`). [`login-amsc.ipynb`](login-amsc.ipynb) checks this.
- **The project must have resources at the site.** If resources were not requested for your project at ESnet or PNNL, the
  site does not allow access to its IRI endpoints (calls are denied, or return no projects/resources).

For access, contact AmSC or request resources for your project on the MyAmSC website: https://my.amsc.energy.gov/

![RIG-UI ESnet / PNNL cards showing Direct via RIG](images/rig-ui/esnet-pnnl-direct.png)

## 3) NERSC (Tier 3: Globus token)

The NERSC IRI API accepts a **Globus token**. RIG does not get it for you: you save it in the Credential Vault, and RIG
sends it to NERSC for calls made in your project.

1. **Get NERSC authorization.** NERSC must authorize your Globus identity for IRI API access.
   Submit a ticket to **TODO: NERSC ticket link and what to request**.
2. **Get a Globus token** with [`iri/login-globus.ipynb`](../iri/login-globus.ipynb). It prints the IRI API token and its expiry (and saves it to
   `~/.iri_token.json`). That notebook needs `GLOBUS_ID` / `GLOBUS_SECRET`; see its header.
3. In the Credential Vault, on the **NERSC** card, paste the token into **Bearer token** and click **Save**
   (**Rotate** if a token is already saved). Then click **Verify via RIG**.
4. Globus tokens are short-lived. When the token expires, repeat steps 2 and 3 and **Rotate** the saved token.

![RIG-UI NERSC card: paste Globus token, Save](images/rig-ui/nersc-save-token.png)

## 4) ALCF (Tier 2: ALCF login)

ALCF uses your **ALCF credentials**. You log in once from the Credential Vault; RIG stores the resulting token and
refreshes it automatically before each API call.

1. You need an ALCF account: https://www.alcf.anl.gov/onboarding-your-project/account-setup
2. In the Credential Vault, on the **ALCF** card, click **Get Auth URL**. A Globus login page opens in a new tab (if a
   pop-up blocker stops it, use the link shown on the card).
3. Log in with your **ALCF credentials** to **Argonne LCF** and copy the authorization code that Globus shows.
4. Paste it into **Authorization Code** and click **Submit Code**. The card changes to **Connected**.

**Disconnect** on the card removes the stored ALCF token.

![Globus login with ALCF credentials selection](images/rig-ui/alcf_globus_login.png)

![Globus login with ALCF credentials and the authorization code](images/rig-ui/alcf-globus-code.png)

![RIG-UI ALCF card: Connected](images/rig-ui/alcf-connected.png)

## 5) ORNL / OLCF (Tier 3: S3M token)

OLCF is reached through the OLCF Secure Scientific Service Mesh (S3M), which takes an **S3M token** from ORNL.
`olcf-open` and `olcf-moderate` are separate enclaves, each with its own token.

1. Get an S3M token for the enclave from the ORNL website. See the OLCF S3M documentation:
   https://docs.olcf.ornl.gov/services_and_applications/s3m/overview.html
2. In the Credential Vault, on the **OLCF Open** (or **OLCF Moderate**) card, paste the token into **Bearer token** and
   click **Save**. Then click **Verify via RIG**.
3. When the token expires, get a new one and **Rotate** the saved token.

![ORNL website: generate an S3M token](images/rig-ui/ornl-s3m-token.png)

![RIG-UI OLCF Open / Moderate card: paste S3M token, Save](images/rig-ui/olcf-save-token.png)

## 6) Check your access
Calls every facility through RIG with your PAT, using the same call as RIG-UI's **Test with RIG** button (the facility's
`probe_path` from RIG: `/account/projects` by default, `/account/whoami` for OLCF), and shows what to fix.
Uses the PAT saved by [`login-amsc.ipynb`](login-amsc.ipynb) (`~/.amsc_token.json`), or `AMSC_PAT` / `AMSC_RIG_URL`.

In [ ]:
import sys
!{sys.executable} -m pip install requests python-dotenv

In [ ]:
import os
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

home_path = Path("~/.amsc_token.json").expanduser()
token_file = json.loads(home_path.read_text()) if home_path.exists() else {}
PAT = os.environ.get("AMSC_PAT") or token_file.get("AMSC_PAT")
RIG_URL = (os.environ.get("AMSC_RIG_URL") or token_file.get("AMSC_RIG_URL") or "https://rig.staging.american-science-cloud.org").rstrip("/")
if not PAT:
    raise RuntimeError(f"No PAT in AMSC_PAT or {home_path}. Run login-amsc.ipynb first.")
HEADERS = {"Authorization": f"Bearer {PAT}", "Accept": "application/json"}

# Where each facility's setup is described in this notebook.
SETUP_SECTION = {
    "esnet-east": "2) ESnet and PNNL", "esnet-west": "2) ESnet and PNNL", "pnnl": "2) ESnet and PNNL",
    "nersc": "3) NERSC", "alcf": "4) ALCF", "olcf-open": "5) ORNL / OLCF", "olcf-moderate": "5) ORNL / OLCF",
}


def api_prefix(f: dict) -> str:
    metadata_path = f.get("metadata_path") or ""
    if metadata_path.endswith("/facility"):
        return metadata_path[: -len("/facility")]
    return f"/api/{f.get('api_version') or 'v1'}"


def check(f: dict) -> tuple:
    probe_path = f.get("probe_path") or f"{api_prefix(f)}/account/projects"
    try:
        r = requests.get(f"{RIG_URL}/rig/external/{f['name']}{probe_path}", headers=HEADERS, timeout=30)
    except requests.RequestException as exc:
        return "ERROR", str(exc)[:100]
    if r.ok:
        # 200 with no projects: reachable, but the project has no resources at this site.
        try:
            no_projects = r.json() == []
        except ValueError:
            no_projects = False
        return ("NO PROJECTS", "no resources for this project at the site") if no_projects else ("OK", "")
    try:
        body = r.json()
        detail = " | ".join(str(body[k]) for k in ("error", "detail", "hint") if isinstance(body, dict) and body.get(k))
    except ValueError:
        detail = r.text
    return f"HTTP {r.status_code}", (detail or r.text)[:100]


r = requests.get(f"{RIG_URL}/ready", headers=HEADERS, timeout=30)
if not r.ok:
    raise RuntimeError(f"RIG /ready failed: {r.status_code} {r.text[:200]}. Run login-amsc.ipynb again.")
facilities = r.json().get("facilities", [])
with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(check, facilities))

rows = [["Facility", "Tier", "Access", "Setup", "Detail"]]
for f, (access, detail) in zip(facilities, results):
    setup = "" if access == "OK" else SETUP_SECTION.get(f["name"], "see the facility's documentation")
    rows.append([f["name"], str(f.get("tier", "")), access, setup, detail])
widths = [max(len(row[i]) for row in rows) for i in range(len(rows[0]))]
for i, row in enumerate(rows):
    print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))
    if i == 0:
        print("-+-".join("-" * w for w in widths))